# Notebook 04 – Phân lớp loại thời tiết (Classification)

**Bài toán:** Dự đoán WeatherType (5 lớp) từ đặc trưng khí tượng.  

**Thiết lập thực nghiệm:**
- Split: 80/20, stratified, seed=42
- Metric chính: **F1-macro** (vì mất cân bằng lớp)
- Metric phụ: F1-weighted, ROC-AUC (OvR), Accuracy

**Mô hình:**
- Baseline 1: DummyClassifier (most_frequent)
- Baseline 2: LogisticRegression (có scaler)
- Mô hình mạnh 1: RandomForest (n=200, class_weight=balanced)
- Mô hình mạnh 2: XGBoost (n=200)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import matplotlib
matplotlib.use('Agg')
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from src.data.loader import load_config, load_processed_data
from src.features.builder import FeatureBuilder, NUMERIC_FEATURES
from src.models.supervised import WeatherClassifier
from src.evaluation.metrics import classification_metrics
from src.evaluation.report import save_table
from src.visualization import plots

cfg = load_config('../configs/params.yaml')
plots.FIG_DIR = '../outputs/figures'
os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs/tables', exist_ok=True)
os.makedirs('../outputs/models', exist_ok=True)

## 4.1 Chuẩn bị dữ liệu

In [2]:
df = load_processed_data('../data/processed/weather_processed.parquet')
builder = FeatureBuilder(cfg)
feat_cols = [c for c in NUMERIC_FEATURES if c in df.columns] + ['Hour','Month','PrecipType_enc']
clf = WeatherClassifier(cfg)
X, y = clf.prepare_Xy(df, feat_cols, 'WeatherType')
print(f'X: {X.shape}, Classes: {clf.le.classes_}')
print(f'Feature columns: {feat_cols}')

[loader] Loaded processed data: (95150, 16)
X: (95150, 10), Classes: ['Clear' 'Cloudy' 'Foggy' 'Rainy' 'Windy']
Feature columns: ['Temperature (C)', 'Apparent Temperature (C)', 'Humidity', 'Wind Speed (km/h)', 'Wind Bearing (degrees)', 'Visibility (km)', 'Pressure (millibars)', 'Hour', 'Month', 'PrecipType_enc']


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (76120, 10), Test: (19030, 10)


## 4.2 Train & So sánh mô hình

In [4]:
results_df = clf.train_all(X_train, y_train, X_test, y_test)
print('\n=== BẢNG KẾT QUẢ SO SÁNH ===')
print(results_df.to_string(index=False))
save_table(results_df, 'clf_results_comparison', '../outputs/tables')


[supervised] Training Baseline_Dummy...
  F1-macro=0.1778, Acc=0.8005, Time=0.0s

[supervised] Training Baseline_LogReg...


  F1-macro=0.5528, Acc=0.8801, Time=1.2s

[supervised] Training RandomForest...


  F1-macro=0.7248, Acc=0.9037, Time=3.0s

[supervised] Training XGBoost...


  F1-macro=0.7578, Acc=0.9016, Time=1.4s

=== BẢNG KẾT QUẢ SO SÁNH ===
          Model  F1_macro  F1_weighted  Accuracy  ROC_AUC  Train_time_s
        XGBoost    0.7578       0.8830    0.9016   0.9584          1.40
   RandomForest    0.7248       0.8760    0.9037   0.9534          2.98
Baseline_LogReg    0.5528       0.8302    0.8801   0.8612          1.24
 Baseline_Dummy    0.1778       0.7118    0.8005   0.5000          0.00
[report] Saved table: ../outputs/tables\clf_results_comparison.csv


'../outputs/tables\\clf_results_comparison.csv'

In [5]:
plots.plot_model_comparison(results_df, 'F1_macro')

[plots] Saved: ../outputs/figures\clf_model_comparison_f1_macro.png


## 4.3 Classification Report & Confusion Matrix

In [6]:
best_model = 'XGBoost'
print(f'Classification Report – {best_model}:')
print(clf.get_classification_report(best_model, y_test))

Classification Report – XGBoost:
              precision    recall  f1-score   support

       Clear       0.64      0.26      0.36      2039
      Cloudy       0.91      0.98      0.94     15234
       Foggy       0.98      0.99      0.99      1417
       Rainy       0.89      0.36      0.52        22
       Windy       0.97      0.99      0.98       318

    accuracy                           0.90     19030
   macro avg       0.88      0.72      0.76     19030
weighted avg       0.88      0.90      0.88     19030



In [7]:
plots.plot_confusion_matrix_heatmap(
    y_test, clf.results['XGBoost']['y_pred'],
    list(clf.le.classes_), 'XGBoost'
)
plots.plot_confusion_matrix_heatmap(
    y_test, clf.results['RandomForest']['y_pred'],
    list(clf.le.classes_), 'RandomForest'
)

[plots] Saved: ../outputs/figures\clf_confusion_xgboost.png


[plots] Saved: ../outputs/figures\clf_confusion_randomforest.png


In [8]:
clf.save_model('XGBoost', '../outputs/models')

[supervised] Model saved to ../outputs/models\XGBoost.joblib


## 4.4 Phân tích lỗi sâu (Deep Error Analysis)

Rubric G yêu cầu: confusion matrix, dạng sai phổ biến, ≥5 actionable insights.

In [9]:
from src.evaluation.metrics import (
    per_class_error_analysis, misclassification_matrix_pct,
    error_analysis_by_season, error_analysis_by_hour,
    extreme_condition_analysis, generate_actionable_insights,
)

class_names = list(clf.le.classes_)
y_pred_best = clf.results['XGBoost']['y_pred']

### 4.4.1 Tỷ lệ lỗi từng lớp

In [10]:
per_class_df = per_class_error_analysis(y_test, y_pred_best, class_names)
print(per_class_df.to_string(index=False))
save_table(per_class_df, 'error_per_class_analysis', '../outputs/tables')
plots.plot_per_class_error_rate(per_class_df)

 Class  Total_samples  Correct  Wrong  Error_rate(%)  Precision  Recall     F1                       Top_confused_with
 Clear           2039      522   1517          74.40     0.6350  0.2560 0.3649 Cloudy (1,509, 74.0%) | Foggy (8, 0.4%)
 Rainy             22        8     14          63.64     0.8889  0.3636 0.5161    Cloudy (13, 59.1%) | Foggy (1, 4.5%)
Cloudy          15234    14909    325           2.13     0.9065  0.9787 0.9412    Clear (300, 2.0%) | Foggy (15, 0.1%)
 Windy            318      315      3           0.94     0.9692  0.9906 0.9798                        Cloudy (3, 0.9%)
 Foggy           1417     1404     13           0.92     0.9832  0.9908 0.9870     Cloudy (12, 0.8%) | Rainy (1, 0.1%)


[report] Saved table: ../outputs/tables\error_per_class_analysis.csv
[plots] Saved: ../outputs/figures\error_per_class_rate.png


### 4.4.2 Ma trận nhầm lẫn chuẩn hoá (%)

In [11]:
cm_pct_df = misclassification_matrix_pct(y_test, y_pred_best, class_names)
print(cm_pct_df.to_string())
save_table(cm_pct_df, 'error_confusion_matrix_pct', '../outputs/tables')
plots.plot_normalized_confusion_matrix(y_test, y_pred_best, class_names, 'XGBoost')

Predicted  Clear  Cloudy  Foggy  Rainy  Windy
Actual                                       
Clear      25.60   74.01   0.39   0.00   0.00
Cloudy      1.97   97.87   0.10   0.00   0.07
Foggy       0.00    0.85  99.08   0.07   0.00
Rainy       0.00   59.09   4.55  36.36   0.00
Windy       0.00    0.94   0.00   0.00  99.06
[report] Saved table: ../outputs/tables\error_confusion_matrix_pct.csv


[plots] Saved: ../outputs/figures\error_confusion_pct_xgboost.png

### 4.4.3 Phân tích lỗi theo mùa (giao mùa/cực trị)

In [12]:
# Lấy df_test gốc với cột Season, Hour, etc.
_, df_test_rows, _, _ = train_test_split(
    df, df['WeatherType'], test_size=0.2, random_state=42, stratify=df['WeatherType']
)
df_test_reset = df_test_rows.reset_index(drop=True)

season_err = error_analysis_by_season(df_test_reset, y_test, y_pred_best, class_names)
print(season_err.to_string())
save_table(season_err, 'error_by_season', '../outputs/tables')
plots.plot_error_by_season(season_err)

        total  errors  error_rate(%) top_error_pair
Season                                             
Summer   4844     602          12.43   Clear→Cloudy
Fall     4644     531          11.43   Clear→Cloudy
Spring   4854     440           9.06   Clear→Cloudy
Winter   4688     299           6.38   Clear→Cloudy
[report] Saved table: ../outputs/tables\error_by_season.csv
[plots] Saved: ../outputs/figures\error_by_season.png


### 4.4.4 Phân tích lỗi theo giờ

In [13]:
hour_err = error_analysis_by_hour(df_test_reset, y_test, y_pred_best, class_names)
print(hour_err.to_string())
save_table(hour_err, 'error_by_hour', '../outputs/tables')
plots.plot_error_by_hour(hour_err)

      total  errors  error_rate(%) top_error_pair
Hour                                             
0       771     147          19.07   Clear→Cloudy
22      824     147          17.84   Clear→Cloudy
2       815     140          17.18   Clear→Cloudy
21      808     136          16.83   Clear→Cloudy
23      782     129          16.50   Clear→Cloudy
1       831     127          15.28   Clear→Cloudy
3       755      93          12.32   Clear→Cloudy
20      801      95          11.86   Clear→Cloudy
4       796      90          11.31   Clear→Cloudy
19      783      86          10.98   Clear→Cloudy
5       769      84          10.92   Clear→Cloudy
8       781      80          10.24   Clear→Cloudy
7       821      71           8.65   Clear→Cloudy
6       780      67           8.59   Clear→Cloudy
9       793      54           6.81   Clear→Cloudy
10      792      53           6.69   Clear→Cloudy
18      799      51           6.38   Clear→Cloudy
17      805      47           5.84   Clear→Cloudy


[report] Saved table: ../outputs/tables\error_by_hour.csv


[plots] Saved: ../outputs/figures\error_by_hour.png


### 4.4.5 Lỗi ở điều kiện cực trị vs bình thường

In [14]:
extreme_err = extreme_condition_analysis(df_test_reset, y_test, y_pred_best, class_names)
print(extreme_err.to_string(index=False))
save_table(extreme_err, 'error_extreme_conditions', '../outputs/tables')
plots.plot_extreme_condition_errors(extreme_err)

                   Condition  N_samples  N_errors  Error_rate(%) Dominant_class
      Temp > 30°C (very hot)        488        58          11.89         Cloudy
           Normal conditions       2126       217          10.21         Cloudy
      Temp < 0°C (very cold)       2105       206           9.79         Cloudy
 Humidity > 0.9 (very humid)       4332       361           8.33         Cloudy
Wind > 25 km/h (strong wind)        835        35           4.19         Cloudy
Visibility < 3 km (very low)       1304        12           0.92          Foggy
[report] Saved table: ../outputs/tables\error_extreme_conditions.csv


[plots] Saved: ../outputs/figures\error_extreme_conditions.png


## 4.5 Actionable Insights (≥5)

In [15]:
# Load time series results nếu đã chạy notebook 05 trước
ts_results = None
ts_path = '../outputs/tables/ts_forecast_results.csv'
if os.path.exists(ts_path):
    ts_results = pd.read_csv(ts_path)

insights = generate_actionable_insights(
    per_class_df=per_class_df,
    season_df=season_err,
    extreme_df=extreme_err,
    ts_results_df=ts_results,
)

for ins in insights:
    print(f"\n{'─'*50}")
    print(f"  INSIGHT #{ins['id']}: {ins['title']}")
    print(f"{'─'*50}")
    print(f"  Phát hiện: {ins['finding']}")
    print(f"  Hành động: {ins['action']}")

save_table(pd.DataFrame(insights), 'actionable_insights', '../outputs/tables')
plots.plot_actionable_insights_summary(insights)


──────────────────────────────────────────────────
  INSIGHT #1: Lớp 'Clear' có tỷ lệ lỗi cao nhất (74.4%)
──────────────────────────────────────────────────
  Phát hiện: Lớp 'Clear' thường bị nhầm sang: Cloudy (1,509, 74.0%) | Foggy (8, 0.4%). Trong 2,039 mẫu, có 1,517 mẫu bị phân lớp sai.
  Hành động: → Bổ sung đặc trưng phân biệt giữa 'Clear' và lớp hay nhầm. Ví dụ: kết hợp Wind Speed + Visibility + Humidity thành interaction features. Có thể dùng SMOTE chỉ cho lớp này nếu thiếu mẫu, hoặc tăng class_weight.

──────────────────────────────────────────────────
  INSIGHT #2: Mùa 'Summer' có tỷ lệ lỗi cao nhất (12.43%)
──────────────────────────────────────────────────
  Phát hiện: Giao mùa 'Summer' là giai đoạn khó phân loại nhất, nhầm phổ biến: Clear→Cloudy. Nguyên nhân: điều kiện thời tiết chuyển tiếp, ranh giới giữa các loại mờ.
  Hành động: → Thêm feature 'Season' dạng one-hot hoặc cyclical encoding (sin/cos month) vào mô hình. Có thể train mô hình riêng cho từng mùa (ensemble the

[plots] Saved: ../outputs/figures\actionable_insights_summary.png
